<a href="https://colab.research.google.com/github/DrGPCR/NEUR201/blob/main/Unit_2/notebooks/Unit_2_Data_Analysis_Assignment_STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit 2 Data Analysis Assignment — What Does Drug Y Do to a Pain Neuron?

**NEUR 201 — Research Methods & Data Analysis for Cellular Neuroscience**

**Name:** *(double-click to type)*  **Date:** *(double-click to type)*

---

## Background

In class you saw how **lidocaine**, a local anaesthetic, changes a pain neuron. Now you'll test a
compound nobody has characterised yet.

A lab has isolated **Drug Y** from a plant. They know it changes how animals respond to touch and pain,
but not **how**. Your job is to find out what Drug Y does to a pain-sensing neuron (a *nociceptor*), and
which ion channel it acts on.

Remember how an action potential is made: **Na⁺ channels** open first and let sodium rush **in**, then
**K⁺ channels** open and let potassium flow **out**.

## Your experiment

You have two **model pain neurons**, identical except for one thing:

| Neuron | Treatment |
|---|---|
| **Control** | none |
| **Drug Y** | exposed to Drug Y |

You'll use both recording methods:

| Method | You control | You measure | What it tells you |
|---|---|---|---|
| **Current clamp** | the injected **current** | the **voltage** | how the neuron *behaves*: threshold, firing rate |
| **Voltage clamp** | the **voltage** | the **current** | *why* it behaves that way: the Na⁺ and K⁺ currents underneath |

## How this notebook works

**Run every cell in order, from the top** (click a cell, press **Shift + Enter**).

| | What to do |
|---|---|
| **⚙️ Try it** | One per step: a value in the code that **you change and re-run**. |
| **💬 Discussion** | At the very end. Answer using the numbers you found along the way. |

**To submit:** answer the Discussion, click **Runtime → Run all**, then **File → Print → Save as PDF**
and upload the PDF to Canvas.

---

## Step 1 — Set up NEURON and prepare the two neurons [1 pt]

The first cell installs NEURON and defines the tools used in every later step. The second cell prepares
**Drug Y**. Its code is hidden on purpose — finding out what it does is the point of the assignment.

In [ ]:
!pip install neuron --quiet

from neuron import h                # our handle on the simulator
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

h.load_file("stdrun.hoc")           # gives us h.run() and h.tstop
print("NEURON is ready.", h.nrnversion())


def build_neuron(treated=False):
    """Build the model pain neuron. If treated=True, it is exposed to Drug Y."""
    soma = h.Section(name="soma")
    soma.L = 50        # length (micrometres)
    soma.diam = 20     # diameter (micrometres)
    soma.insert("hh")  # sodium, potassium and leak channels
    if treated:
        apply_drug(soma)
    return soma


def count_spikes(v):
    """Count upward crossings of 0 mV = number of action potentials."""
    trace = np.array(list(v))
    return int(np.sum((trace[:-1] < 0) & (trace[1:] >= 0)))


NEURONS = [("Control", False, "gray"), ("Drug Y", True, "crimson")]

In [ ]:
#@title Prepare Drug Y (just run this cell)
_FACTOR = 1.3

def apply_drug(soma):
    for seg in soma:
        seg.hh.gnabar = seg.hh.gnabar * _FACTOR

print("Drug Y prepared.")

## Step 2 — Current clamp: find each neuron's threshold [2 pts]

The **threshold** is the weakest stimulus that makes a neuron fire. You'll find it yourself, the way an
electrophysiologist does at the rig: give a brief 1 ms pulse, see whether the neuron fires, and adjust.
The cell below gives **one pulse to each neuron**, and you set each pulse's strength separately.

> **⚙️ Try it — find each neuron's threshold.** Start both pulses at **0.1 nA**, then raise
> `PULSE_CONTROL` and `PULSE_DRUG` in steps of **0.1 nA** (0.2, 0.3 ...), re-running each time. Write
> down the **first pulse strength at which you see an action potential** in each neuron. Stay at
> **1.0 nA or below**.

In [ ]:
PULSE_CONTROL = 0.10    # ⚙️ Try it — pulse strength for the Control neuron (nA)
PULSE_DRUG = 0.10       # ⚙️ Try it — pulse strength for the Drug Y neuron (nA)

pulses = {"Control": PULSE_CONTROL, "Drug Y": PULSE_DRUG}

fig, axes = plt.subplots(3, 1, figsize=(7, 7.5), sharex=True,
                         gridspec_kw={"height_ratios": [1, 2, 2]})

for ax, (name, treated, colour) in zip(axes[1:], NEURONS):
    soma = build_neuron(treated)
    stim = h.IClamp(soma(0.5))        # current-clamp electrode: we set the CURRENT
    stim.delay = 5                    # pulse starts at 5 ms
    stim.dur = 1                      # and lasts 1 ms
    stim.amp = pulses[name]

    v = h.Vector().record(soma(0.5)._ref_v)
    t = h.Vector().record(h._ref_t)
    i_stim = h.Vector().record(stim._ref_i)

    h.v_init = -65
    h.tstop = 30
    h.run()

    axes[0].plot(list(t), list(i_stim), color=colour, label=name)
    ax.plot(list(t), list(v), color=colour)
    ax.axhline(0, color="black", lw=0.5, ls=":")
    ax.set_ylim(-80, 50)
    ax.set_ylabel("mV")
    ax.set_title("%s neuron — %.2f nA pulse" % (name, pulses[name]), fontsize=10)

    print("%-10s  %.2f nA pulse  ->  %-8s (peak %.1f mV)"
          % (name, pulses[name], "SPIKE" if count_spikes(v) > 0 else "no spike", max(list(v))))

axes[0].set_ylabel("Pulse (nA)")
axes[0].set_title("Current clamp: one 1 ms pulse to each neuron")
axes[0].legend(fontsize=8)
axes[2].set_xlabel("Time (ms)")
plt.tight_layout()
plt.show()

if max(pulses.values()) > 1.0:
    print("\nNote: keep pulses at 1.0 nA or below. Above that, the pulse itself can push the")
    print("voltage past 0 mV and be mistaken for a spike.")

## Step 3 — Current clamp: firing rate [1 pt]

A pain neuron signals *how painful* a stimulus is by *how often* it fires. Here each neuron gets a
**steady** current for 100 ms and we count its spikes. Plotting spike count against current gives a
**frequency–current (f–I) curve**, a standard summary of a neuron's excitability. The panels on the
left show the voltage traces at the **strongest** current in your list.

> **⚙️ Try it — fill in the f–I curve.** Add `0.2`, `0.4`, `0.6` and `1.0` to `steady_currents` and re-run. Keep **0.1 nA** in
> the list; your results table reports the spike count there.

In [ ]:
steady_currents = [0.1, 0.8]    # ⚙️ Try it — steady current strengths to test (nA)

steady_currents = sorted(set(steady_currents))     # sort, and drop any repeats
if max(steady_currents) > 1.0:
    print("*** Check your list: it contains %.1f nA, above the 1.0 nA limit. ***" % max(steady_currents))
    print("*** Did you type a comma instead of a decimal point (0,5 instead of 0.5)? ***\n")
spikes = {}
example = {}

for name, treated, _ in NEURONS:
    spikes[name] = []
    for strength in steady_currents:
        soma = build_neuron(treated)
        stim = h.IClamp(soma(0.5))
        stim.delay = 10
        stim.dur = 100                # a long, steady current
        stim.amp = strength
        v = h.Vector().record(soma(0.5)._ref_v)
        t = h.Vector().record(h._ref_t)

        h.v_init = -65
        h.tstop = 120
        h.run()
        spikes[name].append(count_spikes(v))
        example[name] = (list(t), list(v))     # keeps the last (strongest) run

fig = plt.figure(figsize=(11, 4.5))
for position, (name, _, colour) in enumerate(NEURONS):
    ax = fig.add_subplot(2, 2, position * 2 + 1)
    ax.plot(*example[name], color=colour)
    ax.set_ylim(-80, 50)
    ax.set_ylabel("mV")
    ax.set_title("%s at %.1f nA" % (name, steady_currents[-1]), fontsize=10)
ax.set_xlabel("Time (ms)")

ax = fig.add_subplot(1, 2, 2)
for name, _, colour in NEURONS:
    ax.plot(steady_currents, spikes[name], "o-", color=colour, label=name)
ax.set_xlabel("Steady current (nA)")
ax.set_ylabel("Spikes in 100 ms")
ax.set_title("f–I curve")
ax.legend()
plt.tight_layout()
plt.show()

print(pd.DataFrame(spikes, index=["%.1f nA" % s for s in steady_currents]).T)

## Step 4 — Voltage clamp: which current does Drug Y change? [2 pts]

Current clamp showed you *what* changed. Voltage clamp shows you *why*. We hold each neuron at −65 mV,
step it to `STEP_TO` for 40 ms, and record the **sodium** and **potassium** currents separately.
Current flowing **into** the cell is **negative** (drawn downward); current flowing **out** is
**positive** (drawn upward).

> **⚙️ Try it — test more than one voltage.** Run at `STEP_TO = 0`, then try **−40** and **+20**. Is the
> **same** current changed by Drug Y at every voltage? Set `STEP_TO` back to **0** and re-run before moving on.

In [ ]:
STEP_TO = 0      # ⚙️ Try it — the voltage we force the membrane to (mV)

fig, axes = plt.subplots(3, 1, figsize=(7, 7.5), sharex=True,
                         gridspec_kw={"height_ratios": [1, 2, 2]})

peak_na = {}
peak_k = {}

for name, treated, colour in NEURONS:
    soma = build_neuron(treated)

    vc = h.SEClamp(soma(0.5))          # voltage-clamp electrode: we set the VOLTAGE
    vc.rs = 0.01
    vc.dur1 = 10; vc.amp1 = -65        # hold
    vc.dur2 = 40; vc.amp2 = STEP_TO    # step
    vc.dur3 = 10; vc.amp3 = -65        # return

    v = h.Vector().record(soma(0.5)._ref_v)
    ina = h.Vector().record(soma(0.5)._ref_ina)
    ik = h.Vector().record(soma(0.5)._ref_ik)
    t = h.Vector().record(h._ref_t)

    h.tstop = 60
    h.run()

    axes[0].plot(list(t), list(v), color=colour)
    axes[1].plot(list(t), list(ina), color=colour, label=name)
    axes[2].plot(list(t), list(ik), color=colour, label=name)

    peak_na[name] = np.array(list(ina)).min()    # most negative = biggest inward
    peak_k[name] = np.array(list(ik)).max()      # most positive = biggest outward

axes[0].set_ylabel("Command\nvoltage (mV)")
axes[0].set_title("Voltage clamp: step from -65 to %g mV" % STEP_TO)
axes[1].set_ylabel("Na⁺ current\n(mA/cm²)")
axes[1].set_title("Sodium current", fontsize=10)
axes[2].set_ylabel("K⁺ current\n(mA/cm²)")
axes[2].set_title("Potassium current", fontsize=10)
axes[2].set_xlabel("Time (ms)")
for ax in axes[1:]:
    ax.axhline(0, color="black", lw=0.5)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Step to %+g mV:" % STEP_TO)
for name, _, _ in NEURONS:
    print("   %-10s  peak inward Na+ = %.3f mA/cm²    peak outward K+ = %.3f mA/cm²"
          % (name, peak_na[name], peak_k[name]))

### Your results

Your Step 3 and Step 4 measurements in one table. (Your thresholds from Step 2 are the values you
wrote down.)

In [ ]:
KEY_CURRENT = 0.1     # the steady current reported in the table (nA)

if STEP_TO != 0:
    print("*** STEP_TO is %g mV. Set it back to 0 in Step 4 and re-run that cell first. ***\n" % STEP_TO)
if KEY_CURRENT not in steady_currents:
    print("*** %.1f nA is missing from steady_currents. Add it back in Step 3 and re-run that cell. ***\n" % KEY_CURRENT)

def spikes_at(name, current):
    return spikes[name][steady_currents.index(current)] if current in steady_currents else np.nan

my_results = pd.DataFrame([{
    "Neuron": name,
    "Spikes_at_%.1fnA" % KEY_CURRENT: spikes_at(name, KEY_CURRENT),
    "Step_to_mV": STEP_TO,
    "Peak_Na_mA_cm2": round(peak_na[name], 3),
    "Peak_K_mA_cm2": round(peak_k[name], 3),
} for name, treated, _ in NEURONS])

my_results["Na_change_pct"] = round(100 * (my_results["Peak_Na_mA_cm2"] / peak_na["Control"] - 1), 1) + 0.0
my_results["K_change_pct"] = round(100 * (my_results["Peak_K_mA_cm2"] / peak_k["Control"] - 1), 1) + 0.0

my_results.set_index("Neuron").astype(object).T      # .T flips the table on its side so it is easier to read

### 💬 Discussion

Keep each answer to **2–3 sentences** and use the numbers you found.

1. **Threshold (Step 2).** At what pulse strength did you first see an action potential in the Control
   neuron, and in the Drug Y neuron? Based on this, does Drug Y make the pain neuron
   **harder** to fire, **easier** to fire, or have **no effect**? [1 pt]

2. **Firing rate (Step 3).** At 0.1 nA, a weak stimulus, how many spikes did each neuron fire? Would the Drug
   Y neuron send a **stronger** or **weaker** pain signal to the brain for the same weak stimulus? [1 pt]

3. **Voltage clamp (Step 4).** Report the peak Na⁺ and K⁺ currents at 0 mV and their % change. Which
   channel does Drug Y act on, and what does it do to that channel's current? [1 pt]

4. **Treatment.** An animal has been poisoned with Drug Y. Based on your results, would **lidocaine**
   from class be a sensible treatment? Explain using what each drug does to the Na⁺ current. [1 pt]

**Your answer:** *(double-click here to type)*

>
>
>
>

---
### Nice work — you're done!

Click **Runtime → Run all**, then **File → Print → Save as PDF**, and upload the PDF to Canvas.

This cell is empty on purpose. Do not delete.